# Medical Inventory Sales Prediction - Model Development

This notebook orchestrates the modular end-to-end Machine Learning pipeline using reusable components from `src/`.


## 1. Import Libraries and Load Modules


In [ ]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

REPO_ROOT = Path("..").resolve() if Path("..").joinpath("src").exists() else Path(".").resolve()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from src.data_processing import (
    clean_product_data,
    clean_company_data,
    aggregate_product_by_company,
    merge_company_product_data
)
from src.eda import (
    get_top_demand_products, plot_top_demand_products,
    get_low_demand_products, plot_low_demand_products,
    get_overstocked_products, plot_overstocked_products,
    get_fast_moving_products, plot_fast_moving_products,
    plot_sales_vs_stock,
    get_top_companies_by_sales, plot_top_companies,
    plot_purchase_vs_sales,
    get_high_turnover_companies, get_inventory_hoarding_companies
)
from src.feature_engineering import (
    prepare_ml_dataset,
    split_features_and_target,
    create_train_test_split
)
from src.modeling import (
    scale_features,
    train_linear_models,
    train_base_tree_models,
    tune_random_forest,
    tune_gradient_boosting,
    tune_xgboost,
    train_voting_regressor,
    train_stacking_regressor,
    evaluate_model,
    compare_models,
    plot_subplots,
    plot_model_comparison
)
from src.model_persistence import save_model, save_model_columns, save_dashboard_data

sns.set_theme(style="whitegrid")


## 2. Data Processing & Aggregation


In [ ]:
# Define portable dataset paths (assuming local placement in data/)
data_dir = REPO_ROOT / "data"
product_data_path = data_dir / "Product_Level_Data_Final.csv"
company_data_path = data_dir / "Company_level_data_Final.csv"

# Load & Clean product data (includes stock.dropna())
raw_product_df = pd.read_csv(product_data_path)
stock = clean_product_data(raw_product_df)

# Load & Clean company data
raw_company_df = pd.read_csv(company_data_path)
company = clean_company_data(raw_company_df)

# Aggregate product metrics by company & merge
df_grouped = aggregate_product_by_company(stock)
company_grouped = merge_company_product_data(company, df_grouped)

print("Product Stock Dataset Shape:", stock.shape)
print("Merged Company Dataset Shape:", company_grouped.shape)


## 3. Exploratory Data Analysis


In [ ]:
# Product-Level Insights
top_sales = get_top_demand_products(stock, top_n=10)
fig1 = plot_top_demand_products(top_sales)
plt.show()

low_sales, low_count, low_percent, top_low = get_low_demand_products(stock, top_n=10)
print(f"Low-demand products: {low_count} ({low_percent:.2f}%)")
fig2 = plot_low_demand_products(top_low)
plt.show()

overstock = get_overstocked_products(stock, top_n=10)
fig3 = plot_overstocked_products(overstock)
plt.show()

fast_moving = get_fast_moving_products(stock, top_n=10)
fig4 = plot_fast_moving_products(fast_moving)
plt.show()

fig5 = plot_sales_vs_stock(stock)
plt.show()

# Company-Level Insights
top_companies = get_top_companies_by_sales(company_grouped, top_n=10)
fig6 = plot_top_companies(top_companies)
plt.show()

fig7 = plot_purchase_vs_sales(company_grouped)
plt.show()

fast_companies = get_high_turnover_companies(company_grouped, top_n=10)
hoarding_companies = get_inventory_hoarding_companies(company_grouped, top_n=10)
print("High Turnover Companies Count:", len(fast_companies))
print("Inventory Hoarding Companies Count:", len(hoarding_companies))


## 4. Feature Engineering


In [ ]:
ml_model = prepare_ml_dataset(stock)
X, y = split_features_and_target(ml_model, target_column="SaleTot")

X_train, X_test, y_train, y_test = create_train_test_split(
    X, y, test_size=0.2, random_state=5
)

print("Feature Matrix X Shape:", X.shape)
print("Target Vector y Shape:", y.shape)
print("X_train Shape:", X_train.shape)
print("X_test Shape:", X_test.shape)


## 5. Model Training & Comparison


In [ ]:
results_list = []

# Linear Models
X_train_scaled, X_test_scaled, scaler = scale_features(X_train, X_test)
linear_models = train_linear_models(X_train_scaled, y_train)

for name, model in linear_models.items():
    pred = model.predict(X_test_scaled)
    res = evaluate_model(y_test, pred, model_name=name)
    results_list.append(res)
    plot_subplots(y_test, pred, name)
    plt.show()

# Tree Models
tree_models = train_base_tree_models(X_train, y_train)

for name, model in tree_models.items():
    pred = model.predict(X_test)
    res = evaluate_model(y_test, pred, model_name=name)
    results_list.append(res)
    plot_subplots(y_test, pred, name)
    plt.show()

results_df = compare_models(results_list)
display(results_df)

fig_comp = plot_model_comparison(results_df)
plt.show()


## 6. Hyperparameter Tuning


In [ ]:
# Tune Random Forest
rf_grid = tune_random_forest(X_train, y_train, cv=5)
best_rf = rf_grid.best_estimator_
print("Best RF Parameters:", rf_grid.best_params_)
print("Best RF CV R2:", rf_grid.best_score_)

# Tune Gradient Boosting
gb_grid = tune_gradient_boosting(X_train, y_train, cv=3)
best_gb = gb_grid.best_estimator_
print("Best GB Parameters:", gb_grid.best_params_)
print("Best GB CV R2:", gb_grid.best_score_)

# Tune XGBoost
xg_grid = tune_xgboost(X_train, y_train, cv=5)
best_xgb = xg_grid.best_estimator_
print("Best XGB Parameters:", xg_grid.best_params_)
print("Best XGB CV R2:", xg_grid.best_score_)


## 7. Ensemble Models


In [ ]:
estimators = [('rf', best_rf), ('gb', best_gb), ('xgb', best_xgb)]

# Voting Regressor
voting = train_voting_regressor(estimators=estimators, weights=[1, 4, 2], X_train=X_train, y_train=y_train)
pred_voting = voting.predict(X_test)
res_voting = evaluate_model(y_test, pred_voting, model_name="Voting Regressor")
print("Voting Performance:", res_voting)

# Stacking Regressor
from sklearn.linear_model import LinearRegression
stacking = train_stacking_regressor(estimators=estimators, final_estimator=LinearRegression(), X_train=X_train, y_train=y_train)
pred_stacking = stacking.predict(X_test)
res_stacking = evaluate_model(y_test, pred_stacking, model_name="Stacking Regressor")
print("Stacking Performance:", res_stacking)


## 8. Final Model Selection & Persistence


In [ ]:
# Final Model Selection: Gradient Boosting
final_model = best_gb
final_pred = final_model.predict(X_test)
final_metrics = evaluate_model(y_test, final_pred, model_name="Final Gradient Boosting")
print("Final Model Metrics:", final_metrics)

plot_subplots(y_test, final_pred, "Final Gradient Boosting")
plt.show()

# Save Model, Column, and Dashboard Data Artifacts
save_model(final_model, "models/medical_inventory_gb_model.pkl")
save_model_columns(X_train.columns.tolist(), "models/model_columns.pkl")
save_dashboard_data(stock, "models/dashboard_data.pkl")
print("Saved final model artifacts successfully.")